In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

num_borrowers = 4000
num_applications = 8500

borrower_ids = [f"BW-{100000 + i}" for i in range(num_borrowers)]
credit_scores = np.clip(
    np.random.normal(loc=670, scale=60, size=num_borrowers).astype(int),
    300,
    850,
)
annual_incomes = np.clip(
    np.random.lognormal(mean=11.0, sigma=0.5, size=num_borrowers), 20000, 300000
)
emp_length = np.random.randint(0, 16, size=num_borrowers)

df_borrowers_clean = pd.DataFrame(
    {
        "BorrowerID": borrower_ids,
        "CreditScore": credit_scores,
        "AnnualIncome": np.round(annual_incomes, 2),
        "EmploymentLengthYears": emp_length,
    }
)

first_apps = borrower_ids.copy()
remaining_apps = list(
    np.random.choice(borrower_ids, size=num_applications - num_borrowers)
)
all_app_borrowers = first_apps + remaining_apps
np.random.shuffle(all_app_borrowers)

df_loans = pd.DataFrame({"BorrowerID": all_app_borrowers})
df_loans["LoanID"] = [f"LN-{500000 + i}" for i in range(num_applications)]
df_loans = df_loans.merge(df_borrowers_clean, on="BorrowerID", how="left")

all_days = pd.date_range(start="2022-01-01", end="2025-06-30", freq="D")

seasonal_weights = np.array(
    [1.3 if day.month in [10, 11, 12] else 0.85 for day in all_days]
)
seasonal_weights = seasonal_weights / seasonal_weights.sum()

df_loans["IssueDate"] = np.random.choice(
    all_days, size=num_applications, p=seasonal_weights
)
df_loans["IssueDate"] = pd.to_datetime(df_loans["IssueDate"])

df_loans["DebtToIncomeRatio"] = np.round(
    np.clip(np.random.normal(loc=20, scale=8, size=num_applications), 5, 55),
    2,
)
df_loans["TermMonths"] = np.random.choice(
    [36, 60], size=num_applications, p=[0.65, 0.35]
)

requested_base = df_loans["AnnualIncome"] * np.random.uniform(
    0.10, 0.60, size=num_applications
)
df_loans["RequestedAmount"] = np.round(
    np.clip(requested_base, 2000, 50000), 2
)

max_allowed = df_loans["AnnualIncome"] * 0.40

status_list = []
approved_amounts = []

for req, cap in zip(df_loans["RequestedAmount"], max_allowed):
    if req <= cap:
        status_list.append("Approved")
        approved_amounts.append(req)
    elif cap >= 2000:
        status_list.append("Partially_Approved")
        approved_amounts.append(np.round(cap, 2))
    else:
        status_list.append("Rejected")
        approved_amounts.append(0.00)

df_loans["ApplicationStatus"] = status_list
df_loans["ApprovedAmount"] = approved_amounts

df_originated = df_loans[
    df_loans["ApplicationStatus"].isin(["Approved", "Partially_Approved"])
].copy()

risk_credit = (850 - df_originated["CreditScore"]) / 550.0
risk_dti = df_originated["DebtToIncomeRatio"] / 55.0
risk_income = np.maximum(0.0, 1.0 - (df_originated["AnnualIncome"] / 150000.0))
risk_term = np.where(df_originated["TermMonths"] == 60, 0.08, 0.0)

synthetic_risk_score = (
    (risk_credit * 0.45)
    + (risk_dti * 0.25)
    + (risk_income * 0.20)
    + (risk_term * 0.10)
)
default_prob = 1 / (1 + np.exp(-4 * (synthetic_risk_score - 0.45)))
df_originated["IsDefault"] = np.random.binomial(
    1, np.clip(default_prob, 0.01, 0.85)
)

base_apr = 6.0 + (risk_credit * 16.0) + (risk_dti * 5.0) + (risk_term * 20.0)
df_originated["InterestRate"] = np.round(
    np.clip(base_apr + np.random.normal(0, 0.5, len(df_originated)), 5.0, 32.0),
    2,
)

df_loans = df_loans.merge(
    df_originated[["LoanID", "IsDefault", "InterestRate"]],
    on="LoanID",
    how="left",
)
df_loans["IsDefault"] = df_loans["IsDefault"].fillna(0).astype(int)
df_loans.loc[df_loans["ApplicationStatus"] == "Rejected", "InterestRate"] = np.nan


performance_snapshots = []
max_observation_date = pd.to_datetime("2026-06-01")

for _, loan in df_originated.iterrows():
    loan_id = loan["LoanID"]
    principal = loan["ApprovedAmount"]
    rate = loan["InterestRate"] / 100.0 / 12.0
    term = loan["TermMonths"]
    orig_date = loan["IssueDate"]
    is_default = loan["IsDefault"]

    if rate > 0:
        monthly_pmt = principal * (rate * (1 + rate) ** term) / ((1 + rate) ** term - 1)
    else:
        monthly_pmt = principal / term

    
    if is_default == 1:
        risk_factor = synthetic_risk_score.loc[loan.name]
        default_event_month = int(
            np.clip(np.random.normal(loc=18 - (risk_factor * 12), scale=3), 3, min(24, term))
        )
    else:
        default_event_month = -1

    current_balance = principal
    
    for mob in range(1, term + 1):
        snapshot_date = orig_date + pd.DateOffset(months=mob)
        if snapshot_date > max_observation_date:
            break  

        if is_default == 1 and mob >= default_event_month:
            months_since_trigger = mob - default_event_month
            if months_since_trigger == 0:
                dpd = 30
                status = "Delinquent"
            elif months_since_trigger == 1:
                dpd = 60
                status = "Delinquent"
            else:
                dpd = 90
                status = "Defaulted"
                
            performance_snapshots.append(
                {
                    "LoanID": loan_id,
                    "SnapshotDate": snapshot_date.strftime("%Y-%m-%d"),
                    "MonthsOnBook": mob,
                    "DaysPastDue": dpd,
                    "OutstandingPrincipal": np.round(current_balance, 2),
                    "MonthlyPaymentDue": np.round(monthly_pmt, 2),
                    "PerformanceStatus": status,
                }
            )
            if status == "Defaulted":
                break  

        else:
            interest_pmt = current_balance * rate
            principal_pmt = min(current_balance, monthly_pmt - interest_pmt)
            current_balance = max(0.0, current_balance - principal_pmt)
            dpd = 0
            
            if current_balance == 0.0:
                status = "Paid_Off"
                performance_snapshots.append(
                    {
                        "LoanID": loan_id,
                        "SnapshotDate": snapshot_date.strftime("%Y-%m-%d"),
                        "MonthsOnBook": mob,
                        "DaysPastDue": dpd,
                        "OutstandingPrincipal": 0.00,
                        "MonthlyPaymentDue": np.round(monthly_pmt, 2),
                        "PerformanceStatus": status,
                    }
                )
                break  
            else:
                status = "Current"
                performance_snapshots.append(
                    {
                        "LoanID": loan_id,
                        "SnapshotDate": snapshot_date.strftime("%Y-%m-%d"),
                        "MonthsOnBook": mob,
                        "DaysPastDue": dpd,
                        "OutstandingPrincipal": np.round(current_balance, 2),
                        "MonthlyPaymentDue": np.round(monthly_pmt, 2),
                        "PerformanceStatus": status,
                    }
                )

df_performance = pd.DataFrame(performance_snapshots)
df_loans["IssueDate"] = df_loans["IssueDate"].dt.strftime("%Y-%m-%d")

df_borrowers_raw = df_borrowers_clean.copy()
df_loans_raw = df_loans[
    [
        "LoanID",
        "BorrowerID",
        "IssueDate",
        "RequestedAmount",
        "ApprovedAmount",
        "ApplicationStatus",
        "InterestRate",
        "DebtToIncomeRatio",
        "TermMonths",
        "IsDefault",
    ]
].copy()

mask_missing_income = np.random.choice([True, False], size=num_borrowers, p=[0.05, 0.95])
df_borrowers_raw.loc[mask_missing_income, "AnnualIncome"] = np.nan

mask_outlier_dti = np.random.choice([True, False], size=num_applications, p=[0.02, 0.98])
df_loans_raw.loc[mask_outlier_dti, "DebtToIncomeRatio"] = np.round(
    np.random.uniform(150, 300, size=mask_outlier_dti.sum()), 2
)


df_borrowers_raw.to_csv("DimBorrower_Raw.csv", index=False)
df_loans_raw.to_csv("FactLoanApplication_Raw.csv", index=False)
df_performance.to_csv("FactLoanPerformance_Raw.csv", index=False)

print("--- FINTECH ENTERPRISE DATASET V11.1 FROZEN & EXPORTED ---")
print(f"Total Borrowers: {df_borrowers_raw['BorrowerID'].nunique():,}")
print(f"Total Applications: {len(df_loans_raw):,}")
print(f"Originated Loans: {len(df_originated):,}")
print(f"Total Performance Snapshots: {len(df_performance):,}")

In [ ]:
import pandas as pd
import numpy as np


df_borrowers = pd.read_csv(r"D:\PYTHON LEARNING AND PROJECTS\projects\finance project\raw\DimBorrower_Raw.csv")
df_apps = pd.read_csv(r"D:\PYTHON LEARNING AND PROJECTS\projects\finance project\raw\FactLoanApplication_Raw.csv")
df_perf = pd.read_csv(r"D:\PYTHON LEARNING AND PROJECTS\projects\finance project\raw\FactLoanPerformance_Raw.csv")


print("=== MISSING VALUE AUDIT ===")
print("\nDimBorrower Nulls:")
print(df_borrowers.isnull().sum())

print("\nFactLoanApplication Nulls:")
print(df_apps.isnull().sum())

print("\nFactLoanPerformance Nulls:")
print(df_perf.isnull().sum())

print("\n=== DTI EXTREME OUTLIERS AUDIT ===")
dti_outliers = df_apps[df_apps["DebtToIncomeRatio"] > 100]
print(f"Number of corrupted DTI records (>100%): {len(dti_outliers)}")
print(dti_outliers[["LoanID", "DebtToIncomeRatio", "ApplicationStatus"]].head())

=== MISSING VALUE AUDIT ===

DimBorrower Nulls:
BorrowerID                 0
CreditScore                0
AnnualIncome             194
EmploymentLengthYears      0
dtype: int64

FactLoanApplication Nulls:
LoanID               0
BorrowerID           0
IssueDate            0
RequestedAmount      0
ApprovedAmount       0
ApplicationStatus    0
InterestRate         0
DebtToIncomeRatio    0
TermMonths           0
IsDefault            0
dtype: int64

FactLoanPerformance Nulls:
LoanID                  0
SnapshotDate            0
MonthsOnBook            0
DaysPastDue             0
OutstandingPrincipal    0
MonthlyPaymentDue       0
PerformanceStatus       0
dtype: int64

=== DTI EXTREME OUTLIERS AUDIT ===
Number of corrupted DTI records (>100%): 182
        LoanID  DebtToIncomeRatio   ApplicationStatus
52   LN-500052             251.08            Approved
78   LN-500078             284.76            Approved
98   LN-500098             182.77            Approved
168  LN-500168             203.2

In [ ]:
median_income_by_exp = df_borrowers.groupby("EmploymentLengthYears")["AnnualIncome"].transform("median")

df_borrowers["AnnualIncome"] = df_borrowers["AnnualIncome"].fillna(median_income_by_exp)

print(f"Remaining Nulls in AnnualIncome: {df_borrowers['AnnualIncome'].isnull().sum()}")

upper_limit_dti = 55.00

outliers_count = (df_apps["DebtToIncomeRatio"] > 100).sum()

df_apps["DebtToIncomeRatio"] = np.where(
    df_apps["DebtToIncomeRatio"] > 100,   upper_limit_dti,  df_apps["DebtToIncomeRatio"])


print(f"Corrupted DTI records treated: {outliers_count}")
print(f"Max DTI after capping: {df_apps['DebtToIncomeRatio'].max()}%")

In [ ]:
import urllib
from sqlalchemy import create_engine

server = r"KIRANMK\KIRAN_MK"
database = "Finance" 

connection_string = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

params = urllib.parse.quote_plus(connection_string)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")


print("Starting database ingestion into SQL Server...")


df_borrowers.to_sql(
    name="DimBorrower",
    con=engine,
    if_exists="replace",  
    index=False,
    chunksize=1000       
)
print("DimBorrower successfully loaded into SQL Server.")

df_apps.to_sql(
    name="FactLoanApplication",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)
print("FactLoanApplication successfully loaded into SQL Server.")


df_perf.to_sql(
    name="FactLoanPerformance",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000        
)
print(" FactLoanPerformance successfully loaded into SQL Server.")

print("\n--- ALL TABLES SUCCESSFULLY INGESTED INTO SQL SERVER ---")

0       56722.740
1       60883.760
2       57388.485
3       56897.270
4       60302.685
          ...    
3995    60883.760
3996    58559.020
3997    60330.400
3998    62126.740
3999    57500.060
Name: AnnualIncome, Length: 4000, dtype: float64

In [8]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)']
